In [1]:
# =========================
# Food-101 + MobileNetV3-Large
# Train (2-stage) -> Best .keras -> Export SavedModel -> TFLite (FP32/FP16)
# Colab / A100 / T4 공용 "정답 루트"
# =========================

import os
import math
import tensorflow as tf
import tensorflow_datasets as tfds

# -------------------------
# 0) 런타임/성능 기본 설정
# -------------------------
SEED = 42
tf.keras.utils.set_random_seed(SEED)

# 혼합정밀도(학습 속도↑). 변환 자체에는 영향 거의 없지만,
# export/convert 전에 불필요한 XLA는 켜지지 않게 두는 편이 안전합니다.
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")

print("TF:", tf.__version__)
print("Mixed precision policy:", mixed_precision.global_policy())

AUTOTUNE = tf.data.AUTOTUNE


TF: 2.19.0
Mixed precision policy: <DTypePolicy "mixed_float16">


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

...

Shuffling /root/tensorflow_datasets/food101/incomplete.HBRPOI_2.0.0/food101-train.tfrecord*...:   0%|         …

Generating validation examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/food101/incomplete.HBRPOI_2.0.0/food101-validation.tfrecord*...:   0%|    …

Dataset food101 downloaded and prepared to /root/tensorflow_datasets/food101/2.0.0. Subsequent calls will reuse this data.
Train: 75750 Val: 25250
12683000/12683000 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


TypeError: SparseCategoricalCrossentropy.__init__() got an unexpected keyword argument 'label_smoothing'

In [ ]:
# -------------------------
# 1) 하이퍼파라미터 (A100 기준 추천)
# -------------------------
IMG_SIZE = 224          # Android 추론과 동일하게 유지 추천
NUM_CLASSES = 101

# A100이면 보통 224 기준 BATCH 256~512까지 가능.
# 안정성을 위해 256 기본 추천
BATCH_SIZE = 256

# 1단계(Head) + 2단계(Fine-tune)로 나눔
EPOCHS_HEAD = 10
EPOCHS_FT   = 25

# Fine-tune 때 풀 레이어 수 (너무 많이 풀면 과적합/불안정)
UNFREEZE_LAST_N = 40

# label smoothing: 일반적으로 generalization 개선
LABEL_SMOOTHING = 0.1

In [ ]:
# -------------------------
# 2) TFDS Food101 로드 (공식 split 사용)
# -------------------------
# Food101은 tfds에 train/validation split이 존재합니다.
(ds_train, ds_val), ds_info = tfds.load(
    "food101",
    split=["train", "validation"],
    as_supervised=True,
    with_info=True
)

print("Train:", ds_info.splits["train"].num_examples,
      "Val:", ds_info.splits["validation"].num_examples)
class_names = ds_info.features["label"].names  # 101개 클래스 이름

In [ ]:

# -------------------------
# 3) 전처리/증강 (중요: 학습/추론 전처리 일치)
# -------------------------
# MobileNetV3 권장 전처리: preprocess_input을 사용해 [-1, 1] 스케일로 맞춤
# -> Android에서도 동일하게 맞추려면 NormalizeOp(127.5, 127.5) 사용하면 됩니다.
preprocess_input = tf.keras.applications.mobilenet_v3.preprocess_input

def decode_resize(image, label):
    # image: uint8 [0,255]
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE), method="bilinear")
    image = tf.cast(image, tf.float32)
    return image, label

def augment(image, label):
    # 가벼운 증강(과한 RandAugment는 MobileNet에서 오히려 불안정해질 수 있어 기본은 절제)
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.10)
    image = tf.image.random_contrast(image, lower=0.9, upper=1.1)
    # 약간의 랜덤 크롭 느낌 (패딩 후 크롭)
    pad = int(IMG_SIZE * 0.10)
    image = tf.image.resize_with_crop_or_pad(image, IMG_SIZE + pad, IMG_SIZE + pad)
    image = tf.image.random_crop(image, size=[IMG_SIZE, IMG_SIZE, 3])
    return image, label

def to_model_input(image, label):
    # MobileNetV3 preprocess_input: [-1,1]로 정규화
    image = preprocess_input(image)
    return image, label

def build_pipeline(ds, training: bool):
    ds = ds.map(decode_resize, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(8192, seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    ds = ds.map(to_model_input, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE, drop_remainder=training)
    ds = ds.prefetch(AUTOTUNE)
    return ds

train_ds = build_pipeline(ds_train, training=True)
val_ds   = build_pipeline(ds_val,   training=False)

In [ ]:
# -------------------------
# 4) 모델 구성 (MobileNetV3-Large + 새 분류기 Head)
# -------------------------
base = tf.keras.applications.MobileNetV3Large(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    pooling=None
)
base.trainable = False  # 1단계: backbone freeze

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32, name="image")
x = base(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D(name="gap")(x)
x = tf.keras.layers.Dropout(0.2, name="drop")(x)
# Dense는 fp16에서 수치 불안정할 수 있어 dtype float32로 고정 (매우 중요)
outputs = tf.keras.layers.Dense(NUM_CLASSES, dtype=tf.float32, name="logits")(x)

model = tf.keras.Model(inputs, outputs, name="food101_mnv3large")

In [2]:
# -------------------------
# 5) 손실/옵티마이저/콜백
# -------------------------
LABEL_SMOOTHING = 0.1
NUM_CLASSES = 101

def sparse_ce_with_label_smoothing(y_true, y_pred):
    # y_true: (batch,) int
    y_true = tf.cast(y_true, tf.int32)
    y_true_oh = tf.one_hot(y_true, depth=NUM_CLASSES)  # (batch, 101)

    # label smoothing
    ls = tf.constant(LABEL_SMOOTHING, dtype=tf.float32)
    y_true_oh = tf.cast(y_true_oh, tf.float32)
    y_smooth = y_true_oh * (1.0 - ls) + (ls / tf.cast(NUM_CLASSES, tf.float32))

    # categorical cross-entropy (from logits)
    return tf.keras.losses.categorical_crossentropy(y_smooth, y_pred, from_logits=True)

loss_fn = sparse_ce_with_label_smoothing

# AdamW가 있으면 일반화에 유리한 경우가 많음 (TF 버전에 따라 사용 가능)
try:
    optimizer_head = tf.keras.optimizers.AdamW(learning_rate=3e-3, weight_decay=1e-4)
    optimizer_ft   = tf.keras.optimizers.AdamW(learning_rate=1e-4, weight_decay=1e-4)
except Exception:
    optimizer_head = tf.keras.optimizers.Adam(learning_rate=3e-3)
    optimizer_ft   = tf.keras.optimizers.Adam(learning_rate=1e-4)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath="best_food101_mnv3large.keras",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=6,
        mode="max",
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),
]

In [3]:
# -------------------------
# 6) 1단계: Head 학습 (backbone freeze)
# -------------------------
model.compile(
    optimizer=optimizer_head,
    loss=loss_fn,
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5"),
    ],
)

print("\n[Stage 1] Train head only")
history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    callbacks=callbacks
)


[Stage 1] Train head only
Epoch 1/10
294/295 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.3874 - loss: 3.0951 - top5: 0.6275
Epoch 1: val_accuracy improved from -inf to 0.67556, saving model to best_food101_mnv3large.keras
295/295 ━━━━━━━━━━━━━━━━━━━━ 153s 304ms/step - accuracy: 0.3882 - loss: 3.0919 - top5: 0.6283 - val_accuracy: 0.6756 - val_loss: 1.9602 - val_top5: 0.8910 - learning_rate: 0.0030
Epoch 2/10
294/295 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.5982 - loss: 2.2355 - top5: 0.8381
Epoch 2: val_accuracy improved from 0.67556 to 0.68558, saving model to best_food101_mnv3large.keras
295/295 ━━━━━━━━━━━━━━━━━━━━ 33s 106ms/step - accuracy: 0.5982 - loss: 2.2355 - top5: 0.8381 - val_accuracy: 0.6856 - val_loss: 1.9187 - val_top5: 0.8990 - learning_rate: 0.0030
Epoch 3/10
294/295 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.6201 - loss: 2.1641 - top5: 0.8495
Epoch 3: val_accuracy improved from 0.68558 to 0.68653, saving model to best_food101_mnv3large.keras
295/295 ━━

In [4]:
# -------------------------
# 7) 2단계: Fine-tuning (backbone 일부 해제)
# -------------------------
base.trainable = True

# 상위 일부만 trainable (과적합/불안정 방지)
for layer in base.layers[:-UNFREEZE_LAST_N]:
    layer.trainable = False
for layer in base.layers[-UNFREEZE_LAST_N:]:
    layer.trainable = True

model.compile(
    optimizer=optimizer_ft,
    loss=loss_fn,
    metrics=[
        tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5"),
    ],
)

print("\n[Stage 2] Fine-tune last N layers:", UNFREEZE_LAST_N)
history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FT,
    callbacks=callbacks
)

# 베스트 모델 로드
best_model = tf.keras.models.load_model("best_food101_mnv3large.keras", compile=False)




[Stage 2] Fine-tune last N layers: 40
Epoch 1/25
294/295 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.5857 - loss: 2.2764 - top5: 0.8236
Epoch 1: val_accuracy did not improve from 0.71192
295/295 ━━━━━━━━━━━━━━━━━━━━ 95s 175ms/step - accuracy: 0.5861 - loss: 2.2749 - top5: 0.8239 - val_accuracy: 0.5969 - val_loss: 2.2723 - val_top5: 0.8512 - learning_rate: 1.0000e-04
Epoch 2/25
294/295 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.7084 - loss: 1.8464 - top5: 0.9072
Epoch 2: val_accuracy did not improve from 0.71192
295/295 ━━━━━━━━━━━━━━━━━━━━ 33s 103ms/step - accuracy: 0.7085 - loss: 1.8461 - top5: 0.9072 - val_accuracy: 0.6787 - val_loss: 1.9600 - val_top5: 0.8979 - learning_rate: 1.0000e-04
Epoch 3/25
294/295 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.7387 - loss: 1.7495 - top5: 0.9219
Epoch 3: val_accuracy improved from 0.71192 to 0.73255, saving model to best_food101_mnv3large.keras
295/295 ━━━━━━━━━━━━━━━━━━━━ 33s 105ms/step - accuracy: 0.7387 - loss: 1.7493 - top5: 0.

In [5]:
# -------------------------
# 8) SavedModel Export (TFLite/Serving용)
# Keras 3 권장: model.export(dir)
# -------------------------
EXPORT_DIR = "food101_mnv3large_export"
if os.path.exists(EXPORT_DIR):
    # Colab에서 재실행 대비
    import shutil
    shutil.rmtree(EXPORT_DIR)

best_model.export(EXPORT_DIR)
print("Exported SavedModel:", EXPORT_DIR)

Saved artifact at 'food101_mnv3large_export'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): List[TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='image')]
Output Type:
  TensorSpec(shape=(None, 101), dtype=tf.float32, name=None)
Captures:
  137153339365264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137153339366224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137153339366032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137153339365648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137153339363536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137153339365840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137153339363728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137153339361040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137153339364496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137153339364112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13715

In [8]:
# -------------------------
# 9) TFLite 변환 (Builtins-only 우선)
# -------------------------
def convert_tflite(export_dir: str, fp16: bool = False) -> bytes:
    converter = tf.lite.TFLiteConverter.from_saved_model(export_dir)

    # 1) 기본: Builtins-only로 최대한 깨끗하게
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]

    # 2) FP16 변환 옵션
    if fp16:
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]
    else:
        # FP32는 변환 안정성이 우선이라 최적화 옵션을 일단 끕니다.
        # (필요하면 나중에 켜서 테스트)
        converter.optimizations = []

    # 3) 안정화 옵션 (환경에 따라 도움)
    converter.experimental_enable_resource_variables = True

    # 4) 1차 시도 (Builtins-only)
    try:
        return converter.convert()
    except Exception as e:
        # 5) 실패 시: Select TF Ops(Flex) 허용으로 fallback
        #    Android에서 select-tf-ops 의존성이 필요해질 수 있습니다.
        converter.target_spec.supported_ops = [
            tf.lite.OpsSet.TFLITE_BUILTINS,
            tf.lite.OpsSet.SELECT_TF_OPS,
        ]
        return converter.convert()


In [9]:
# FP32
tflite_fp32 = convert_tflite(EXPORT_DIR, fp16=False)
FP32_PATH = "food101_mnv3large_fp32.tflite"
with open(FP32_PATH, "wb") as f:
    f.write(tflite_fp32)
print("FP32 TFLite (MB):", len(tflite_fp32) / (1024 * 1024))

# FP16
tflite_fp16 = convert_tflite(EXPORT_DIR, fp16=True)
FP16_PATH = "food101_mnv3large_fp16.tflite"
with open(FP16_PATH, "wb") as f:
    f.write(tflite_fp16)
print("FP16 TFLite (MB):", len(tflite_fp16) / (1024 * 1024))

FP32 TFLite (MB): 6.2167816162109375
FP16 TFLite (MB): 5.990253448486328


In [10]:
# -------------------------
# 10) 간단 검증: TFLite로 실제 추론이 "한 클래스만" 나오지 않는지 확인
# -------------------------
import numpy as np

interpreter = tf.lite.Interpreter(model_path=FP32_PATH)
interpreter.allocate_tensors()
inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [11]:
def tflite_predict_one(batch_images):
    # batch_images: float32, shape [B, IMG, IMG, 3], 이미 preprocess_input 적용된 값
    interpreter.set_tensor(inp["index"], batch_images)
    interpreter.invoke()
    logits = interpreter.get_tensor(out["index"])
    pred = np.argmax(logits, axis=-1)
    return pred, logits

In [13]:
import numpy as np

def tflite_predict_one(images_np):
    """
    images_np: (B, H, W, 3) 또는 (H, W, 3)
    반환: pred (B,), logits/prob (B,101)
    """
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # (H,W,3) 들어오면 배치 1로
    if images_np.ndim == 3:
        images_np = images_np[None, ...]

    # 모델이 배치 1 고정이면, B개를 for-loop로 1장씩 처리
    B = images_np.shape[0]
    preds = []
    outs = []

    for i in range(B):
        x = images_np[i:i+1].astype(np.float32)  # (1,H,W,3)
        interpreter.set_tensor(input_details[0]["index"], x)
        interpreter.invoke()
        y = interpreter.get_tensor(output_details[0]["index"])  # (1,101)
        outs.append(y[0])
        preds.append(int(np.argmax(y[0])))

    return np.array(preds), np.stack(outs, axis=0)


In [14]:
# validation에서 1배치 뽑아 예측 분포 확인
for images, labels in val_ds.take(1):
    images_np = images.numpy().astype(np.float32)
    pred, logits = tflite_predict_one(images_np[:16])
    print("Sample preds:", pred.tolist())
    print("Unique preds in 16:", len(set(pred.tolist())))
    break

Sample preds: [29, 81, 91, 53, 97, 31, 10, 31, 3, 94, 4, 32, 32, 3, 8, 77]
Unique preds in 16: 13


In [ ]:
# -------------------------
# 11) Colab 다운로드 자동화
# -------------------------
from google.colab import files
files.download(FP32_PATH)
files.download(FP16_PATH)
print("Downloaded:", FP32_PATH, FP16_PATH)

In [59]:
import numpy as np
import tensorflow as tf
from PIL import Image, ImageOps

IMG_SIZE = 224
model = tf.keras.models.load_model("best_food101_mnv3large.keras", compile=False)

# val_ds 스케일 자동 감지
x0, _ = next(iter(val_ds.take(1)))
vmin = float(tf.reduce_min(x0))
vmax = float(tf.reduce_max(x0))

if vmax > 10:      # 대략 0~255
    MODE = "raw255"
elif vmin < -0.2:  # 대략 -1~1
    MODE = "m11"
else:              # 대략 0~1
    MODE = "01"

print("Detected MODE from val_ds:", MODE, "(min/max:", vmin, vmax, ")")

def load_image_pil(path):
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)
    img = img.convert("RGB")
    img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    x = np.asarray(img).astype(np.float32)
    return x

def preprocess_by_mode(x, mode):
    if mode == "raw255":
        # 모델/학습이 0~255를 받았던 케이스
        return x
    if mode == "01":
        return x / 255.0
    if mode == "m11":
        return (x - 127.5) / 127.5
    raise ValueError(mode)

def predict_topk(path, k=5):
    x = load_image_pil(path)
    x = preprocess_by_mode(x, MODE)
    x = np.expand_dims(x, 0).astype(np.float32)

    # 중요: 딕셔너리 입력으로 전달
    logits = model({"image": x}, training=False).numpy().squeeze()
    probs = tf.nn.softmax(logits).numpy()
    top = probs.argsort()[::-1][:k]
    return top.tolist(), probs[top].tolist(), float(probs[top[0]])

# 테스트
print("Pred:", predict_topk("food101_sample.jpg", k=5))


Detected MODE from val_ds: raw255 (min/max: 0.0 255.0 )
Pred: ([29, 30, 100, 94, 83], [0.7997840642929077, 0.10344123095273972, 0.013359392061829567, 0.01103781908750534, 0.00453196233138442], 0.7997840642929077)
